import yfinance as yf
print(yf.__version__)

In [1]:
# OHLC + Volume Feature Engineering

In [2]:
# 1. Data Integrity

In [3]:
## 1.1 Pobierz dane

In [4]:
import yfinance as yf
import pandas as pd
from pathlib import Path

# =========================
# Parametry
# =========================
TICKER = "^GSPC"
START_DATE = "1900-01-01"
END_DATE = None
INTERVAL = "1d"

DATA_DIR = Path("data/market")
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILE_PATH = DATA_DIR / f"{TICKER.replace('^','')}_{INTERVAL}.parquet"


def load_or_download_data(
    ticker: str,
    start: str,
    end: str | None,
    interval: str,
    file_path: Path
) -> pd.DataFrame:
    """
    Ładuje dane z dysku jeśli istnieją,
    w przeciwnym razie pobiera je z yfinance i zapisuje.
    """

    if file_path.exists():
        print(f"[INFO] Wczytywanie danych z cache: {file_path}")
        try:
            df = pd.read_parquet(file_path)
            return df
        except Exception as e:
            raise RuntimeError(
                f"Błąd podczas wczytywania danych z {file_path}"
            ) from e

    print("[INFO] Brak danych w cache — pobieranie z yfinance...")
    try:
        df = yf.download(
            tickers=ticker,
            start=start,
            end=end,
            interval=interval,
            auto_adjust=False,
            progress=False
        )
    except Exception as e:
        raise RuntimeError("Błąd podczas pobierania danych z yfinance") from e

    if df.empty:
        raise ValueError("Pobrane dane są puste — sprawdź ticker lub zakres dat")

    # Spłaszczenie MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]

    # Standaryzacja nazw kolumn
    df = df.rename(columns={
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    })

    df = df.dropna()

    try:
        df.to_parquet(file_path)
        print(f"[INFO] Dane zapisane do: {file_path}")
    except Exception as e:
        raise RuntimeError(
            f"Nie udało się zapisać danych do {file_path}"
        ) from e

    return df


# =========================
# Użycie
# =========================
df = load_or_download_data(
    ticker=TICKER,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    file_path=FILE_PATH
)

print(df.head())
print(df.tail())
print(df.info())


[INFO] Brak danych w cache — pobieranie z yfinance...
[INFO] Dane zapisane do: data/market/GSPC_1d.parquet
            adj_close      close       high        low       open  volume
Date                                                                     
1927-12-30  17.660000  17.660000  17.660000  17.660000  17.660000       0
1928-01-03  17.760000  17.760000  17.760000  17.760000  17.760000       0
1928-01-04  17.719999  17.719999  17.719999  17.719999  17.719999       0
1928-01-05  17.549999  17.549999  17.549999  17.549999  17.549999       0
1928-01-06  17.660000  17.660000  17.660000  17.660000  17.660000       0
              adj_close        close         high          low         open  \
Date                                                                          
2026-01-26  6950.229980  6950.229980  6964.660156  6921.600098  6923.229980   
2026-01-27  6978.600098  6978.600098  6988.819824  6958.830078  6965.959961   
2026-01-28  6978.029785  6978.029785  7002.279785  6963.459

In [5]:
### 1.1.1 Utnij dane które są nieistotne - wykazano po sprawdzeniu danych

In [6]:
df = df[df.index >= "1952-01-01"].copy()

In [7]:
### 1.1.2 Wymuszenie polityki timestampów

In [8]:
import pandas as pd

def enforce_daily_session_timestamps(df: pd.DataFrame, *, verbose: bool = True) -> pd.DataFrame:
    """
    Wymusza politykę timestampów:
    - DatetimeIndex
    - timezone-naive
    - normalized to session date (00:00)
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("Index must be pandas.DatetimeIndex")

    df = df.copy()
    idx_before = df.index

    had_tz = idx_before.tz is not None
    had_time_component = not (idx_before == idx_before.normalize()).all()

    if had_tz:
        df.index = df.index.tz_convert(None)

    df.index = df.index.normalize()

    if verbose:
        print("🕒 enforce_daily_session_timestamps:")
        print(f"   - timezone removed: {had_tz}")
        print(f"   - time component removed: {had_time_component}")
        print("   - policy: daily session dates (naive)")

    return df

df = enforce_daily_session_timestamps(df)


🕒 enforce_daily_session_timestamps:
   - timezone removed: False
   - time component removed: False
   - policy: daily session dates (naive)


In [9]:
### 1.1.3 Correct timestamps – twarda walidacja

In [10]:
import pandas as pd

def check_correct_timestamps(df: pd.DataFrame) -> None:
    """
    Waliduje poprawność timestampów dla daily OHLC.
    Rzuca ValueError z czytelnym komunikatem przy pierwszym błędzie.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("Index is not a pandas.DatetimeIndex")

    idx = df.index

    if idx.tz is not None:
        raise ValueError("Index has timezone info; expected timezone-naive daily timestamps")

    if not idx.is_monotonic_increasing:
        raise ValueError("Timestamps are not sorted (not monotonic increasing)")

    if not idx.is_unique:
        raise ValueError("Duplicate timestamps detected")

    if not (idx == idx.normalize()).all():
        bad = idx[idx != idx.normalize()]
        raise ValueError(
            f"Timestamps contain time component; examples: {bad[:5].tolist()}"
        )

    print("✅ Correct timestamps: PASS")

check_correct_timestamps(df)

✅ Correct timestamps: PASS


In [11]:
## 1.2. Sprawdź dane

In [12]:
### 1.2.1 Indeks czasowy
assert isinstance(df.index, pd.DatetimeIndex)
assert df.index.is_monotonic_increasing
assert df.index.is_unique

### 1.2.2 Logika OHLC
assert (df["high"] >= df[["open", "close"]].max(axis=1)).all()
assert (df["low"]  <= df[["open", "close"]].min(axis=1)).all()
assert (df["high"] >= df["low"]).all()

### 1.2.3 Wolumen
assert (df["volume"] >= 0).all()

print("✅ Data Integrity: podstawowe testy zaliczone")


✅ Data Integrity: podstawowe testy zaliczone


In [13]:
import pandas as pd

def check_session_gaps_daily(df: pd.DataFrame, max_gap_days: int = 4):
    """
    Dla danych dziennych wykrywa podejrzane luki między kolejnymi świecami.
    Domyślnie >4 dni jest podejrzane (bo weekend to 2 dni, długi weekend ~3-4).
    """
    assert isinstance(df.index, pd.DatetimeIndex)

    idx = df.index.sort_values().normalize()
    deltas = idx.to_series().diff().dropna()

    suspicious = deltas[deltas > pd.Timedelta(days=max_gap_days)]
    return {
        "n_rows": len(df),
        "max_gap": deltas.max(),
        "n_suspicious_gaps": len(suspicious),
        "suspicious_gaps": suspicious.head(20),  # pokaże daty i wielkość przerwy
    }

# użycie:
report = check_session_gaps_daily(df, max_gap_days=4)
print(report)


{'n_rows': 18643, 'max_gap': Timedelta('7 days 00:00:00'), 'n_suspicious_gaps': 7, 'suspicious_gaps': Date
1956-12-26   5 days
1958-12-29   5 days
1961-05-31   5 days
1968-07-08   5 days
2001-09-17   7 days
2007-01-03   5 days
2012-10-31   5 days
Name: Date, dtype: timedelta64[ns]}


In [14]:
import pandas as pd
import pandas_market_calendars as mcal

def check_continuous_sessions_with_calendar(
    df: pd.DataFrame,
    calendar_name: str = "NYSE",
    *,
    weekdays: tuple[int, ...] = (0, 1, 2, 3, 4),  # 0=Mon ... 6=Sun
    normalize_index: bool = True,
):
    """
    Sprawdza, czy df ma świecę dla każdej oczekiwanej sesji z kalendarza giełdowego.
    
    Parametry:
    - weekdays: które dni tygodnia uznajemy za "potencjalne sesje".
      Domyślnie (0..4) => poniedziałek–piątek.
      Jeśli chcesz dopuścić soboty: (0,1,2,3,4,5)
    - normalize_index: jeśli True, porównuje po datach (bez godzin).
    
    Zwraca słownik z liczbą braków i podglądem brakujących dat.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("df.index musi być pandas.DatetimeIndex")

    idx = df.index
    if normalize_index:
        # dzienne dane z yfinance zwykle są bez TZ; normalizacja to bezpieczny standard
        idx = pd.DatetimeIndex(idx).normalize()

    idx = idx.sort_values()
    idx = idx[~idx.duplicated(keep="first")]

    cal = mcal.get_calendar(calendar_name)
    start = idx.min().date()
    end = idx.max().date()

    schedule = cal.schedule(start_date=start, end_date=end)

    expected_sessions = pd.DatetimeIndex(schedule.index)
    if normalize_index:
        expected_sessions = expected_sessions.normalize()

    # 🔑 Filtrujemy kalendarz do wybranych dni tygodnia (np. Mon–Fri)
    expected_sessions = expected_sessions[expected_sessions.weekday.isin(weekdays)]

    missing_sessions = expected_sessions.difference(idx)
    extra_sessions = idx.difference(expected_sessions)

    # Dodatkowe info diagnostyczne
    return {
        "calendar": calendar_name,
        "weekdays": weekdays,
        "start": str(start),
        "end": str(end),
        "n_expected_sessions": len(expected_sessions),
        "n_actual_sessions": len(idx),
        "n_missing_sessions": len(missing_sessions),
        "missing_sessions_head": missing_sessions[:20],
        "n_extra_sessions": len(extra_sessions),
        "extra_sessions_head": extra_sessions[:20],
    }

report = check_continuous_sessions_with_calendar(df, "NYSE", weekdays=(0,1,2,3,4))
print(report)


{'calendar': 'NYSE', 'weekdays': (0, 1, 2, 3, 4), 'start': '1952-01-02', 'end': '2026-01-30', 'n_expected_sessions': 18643, 'n_actual_sessions': 18643, 'n_missing_sessions': 0, 'missing_sessions_head': DatetimeIndex([], dtype='datetime64[ns]', freq=None), 'n_extra_sessions': 0, 'extra_sessions_head': DatetimeIndex([], dtype='datetime64[ns]', freq=None)}


In [15]:
assert report["n_missing_sessions"] == 0

In [16]:
# 2. Price Transforms

In [ ]:
## 2.1 Log returns

In [17]:
import numpy as np
import pandas as pd

def compute_log_return(close: pd.Series) -> pd.Series:
    close = close.astype(float)
    # log(close_t) - log(close_{t-1})
    return np.log(close).diff()


In [18]:
from sklearn.base import BaseEstimator, TransformerMixin

class LogReturnTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, close_col: str = "close", output_col: str = "log_return", drop_na: bool = True):
        self.close_col = close_col
        self.output_col = output_col
        self.drop_na = drop_na

    def fit(self, X, y=None):
        # Stateless transformer: niczego nie uczymy
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")

        if self.close_col not in X.columns:
            raise ValueError(f"Missing required column: '{self.close_col}'")

        out = pd.DataFrame(index=X.index)
        out[self.output_col] = compute_log_return(X[self.close_col])

        # Pierwszy wiersz będzie NaN (bo diff)
        if self.drop_na:
            out = out.dropna()

        return out


In [19]:
# zakładam, że masz df z kolumnami: open/high/low/close/volume
lrt = LogReturnTransformer(close_col="close", output_col="log_return", drop_na=False)

logret = lrt.fit_transform(df)

print(logret.head(5))
print(logret.describe())

# Sanity checks
assert "log_return" in logret.columns
# 1) Pierwsza świeca ma NaN (brak t-1)
assert pd.isna(logret["log_return"].iloc[0])
# 2) Reszta powinna być skończona (o ile nie masz close <= 0)
assert np.isfinite(logret["log_return"].iloc[1:]).all()

print("✅ LogReturnTransformer: PASS")


            log_return
Date                  
1952-01-02         NaN
1952-01-03    0.003356
1952-01-04    0.001674
1952-01-07   -0.000418
1952-01-08   -0.003771
         log_return
count  18642.000000
mean       0.000304
std        0.010007
min       -0.228997
25%       -0.004079
50%        0.000490
75%        0.005064
max        0.109572
✅ LogReturnTransformer: PASS


In [ ]:
## 2.2 Normalized ranges

In [47]:
full_range = (df["high"].astype(float) - df["low"].astype(float))
print("Zero-range candles:", (full_range <= 1e-12).sum())
print("Examples:", df.index[full_range <= 1e-12][:10].tolist())


Zero-range candles: 2553
Examples: [Timestamp('1952-01-02 00:00:00'), Timestamp('1952-01-03 00:00:00'), Timestamp('1952-01-04 00:00:00'), Timestamp('1952-01-07 00:00:00'), Timestamp('1952-01-08 00:00:00'), Timestamp('1952-01-09 00:00:00'), Timestamp('1952-01-10 00:00:00'), Timestamp('1952-01-11 00:00:00'), Timestamp('1952-01-14 00:00:00'), Timestamp('1952-01-15 00:00:00')]


In [54]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class NormalizedRangeTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        open_col="open",
        high_col="high",
        low_col="low",
        close_col="close",
        eps: float = 1e-12,
        drop_na: bool = True,
        zero_range_policy: str = "nan",  # "nan" albo "zeros"
    ):
        self.open_col = open_col
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col
        self.eps = eps
        self.drop_na = drop_na
        self.zero_range_policy = zero_range_policy

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")

        for col in [self.open_col, self.high_col, self.low_col, self.close_col]:
            if col not in X.columns:
                raise ValueError(f"Missing column: {col}")

        o = X[self.open_col].astype(float)
        h = X[self.high_col].astype(float)
        l = X[self.low_col].astype(float)
        c = X[self.close_col].astype(float)

        full_range_raw = (h - l)
        zero_range = full_range_raw <= self.eps

        # do obliczeń dzielnika używamy bezpiecznego (żeby nie dzielić przez 0)
        full_range = full_range_raw.mask(zero_range, np.nan)

        out = pd.DataFrame(index=X.index)

        # Price transforms (tu nie ma problemu z sumą)
        out["hl_range"] = (h - l) / c
        out["co_return"] = (c - o) / o

        # Geometry: NaN dla zero-range (domyślnie)
        out["body_ratio"] = (c - o).abs() / full_range
        out["upper_wick_ratio"] = (h - np.maximum(o, c)) / full_range
        out["lower_wick_ratio"] = (np.minimum(o, c) - l) / full_range

        if self.zero_range_policy == "zeros":
            out.loc[zero_range, ["body_ratio", "upper_wick_ratio", "lower_wick_ratio"]] = 0.0
        elif self.zero_range_policy == "nan":
            # zostaje NaN
            pass
        else:
            raise ValueError("zero_range_policy must be 'nan' or 'zeros'")

        if self.drop_na:
            out = out.dropna()

        return out


In [68]:
nrt = NormalizedRangeTransformer(drop_na=False, zero_range_policy="nan")
nr = nrt.fit_transform(df)

print('Before normalization: ')
print(df.dropna().head(20))
#print(nr[:20])
print('After normalization: ')
print(nr.dropna().head(20))

Before normalization: 
            adj_close      close       high        low       open   volume
Date                                                                      
1952-01-02  23.799999  23.799999  23.799999  23.799999  23.799999  1070000
1952-01-03  23.879999  23.879999  23.879999  23.879999  23.879999  1220000
1952-01-04  23.920000  23.920000  23.920000  23.920000  23.920000  1480000
1952-01-07  23.910000  23.910000  23.910000  23.910000  23.910000  1540000
1952-01-08  23.820000  23.820000  23.820000  23.820000  23.820000  1390000
1952-01-09  23.740000  23.740000  23.740000  23.740000  23.740000  1370000
1952-01-10  23.860001  23.860001  23.860001  23.860001  23.860001  1520000
1952-01-11  23.980000  23.980000  23.980000  23.980000  23.980000  1760000
1952-01-14  24.160000  24.160000  24.160000  24.160000  24.160000  1510000
1952-01-15  24.059999  24.059999  24.059999  24.059999  24.059999  1340000
1952-01-16  24.090000  24.090000  24.090000  24.090000  24.090000  1430000
19

In [60]:
# maska tych, gdzie ratios są policzalne
mask = nr[["body_ratio", "upper_wick_ratio", "lower_wick_ratio"]].notna().all(axis=1)

assert (nr["hl_range"] >= 0).all()
assert nr.loc[mask, "body_ratio"].between(0, 1).all()

s = (nr.loc[mask, "body_ratio"] +
     nr.loc[mask, "upper_wick_ratio"] +
     nr.loc[mask, "lower_wick_ratio"])

assert np.allclose(s.to_numpy(), 1.0, atol=1e-6)

print(f"✅ Normalized ranges: PASS (checked {mask.sum()} rows, skipped {(~mask).sum()} zero-range/NaN rows)")


✅ Normalized ranges: PASS (checked 16090 rows, skipped 2553 zero-range/NaN rows)


In [61]:
o, h, l, c = (df["open"].astype(float), df["high"].astype(float),
             df["low"].astype(float), df["close"].astype(float))

bad = ~((h >= np.maximum(o, c)) & (l <= np.minimum(o, c)) & (h >= l))
print("Bad OHLC rows:", bad.sum())
print(df.loc[bad].head(10))


Bad OHLC rows: 0
Empty DataFrame
Columns: [adj_close, close, high, low, open, volume]
Index: []


In [62]:
## 2.3 No absolute prices - na koniec feature enginering

In [63]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class DropAbsolutePricesGuard(BaseEstimator, TransformerMixin):
    def __init__(self, forbidden=("open", "high", "low", "close", "adj_close")):
        self.forbidden = tuple(forbidden)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")

        present = [c for c in self.forbidden if c in X.columns]
        if present:
            raise ValueError(
                f"Absolute price columns found (forbidden): {present}. "
                "Build features first and pass only transformed features downstream."
            )
        return X


In [64]:
def validate_no_absolute_prices(features: pd.DataFrame, forbidden=("open","high","low","close","adj_close")) -> dict:
    if not isinstance(features, pd.DataFrame):
        raise TypeError("features must be a DataFrame")

    forbidden = set(forbidden)
    present = sorted(list(forbidden.intersection(features.columns)))

    report = {
        "status": "PASS" if len(present) == 0 else "FAIL",
        "forbidden_present": present,
        "n_features": features.shape[1],
    }
    return report

# użycie:
# report = validate_no_absolute_prices(X_features)
# print(report)
# assert report["status"] == "PASS"


In [ ]:
# 3. Candle Anatomy

In [ ]:
## 3.1 Upper / Lower wick

In [70]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class PriceTransformTransformer(BaseEstimator, TransformerMixin):
    """
    Price Transforms (sekcja 2):
    - log_return (close-to-close)
    - hl_range = (high-low)/close
    - co_return = (close-open)/open
    """
    def __init__(
        self,
        open_col="open",
        high_col="high",
        low_col="low",
        close_col="close",
        eps: float = 1e-12,
        drop_na: bool = False,
    ):
        self.open_col = open_col
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col
        self.eps = eps
        self.drop_na = drop_na

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")

        for col in [self.open_col, self.high_col, self.low_col, self.close_col]:
            if col not in X.columns:
                raise ValueError(f"Missing column: {col}")

        o = X[self.open_col].astype(float)
        h = X[self.high_col].astype(float)
        l = X[self.low_col].astype(float)
        c = X[self.close_col].astype(float)

        if (c <= 0).any() or (o <= 0).any():
            raise ValueError("Found non-positive open/close values; cannot compute log_return or co_return safely.")

        out = pd.DataFrame(index=X.index)

        # close-to-close log return (uses t and t-1 only)
        out["log_return"] = np.log(c).diff()

        # normalized candle range (stateless)
        out["hl_range"] = (h - l) / c

        # open-to-close return (stateless)
        out["co_return"] = (c - o) / o

        if self.drop_na:
            out = out.dropna()

        return out


In [71]:
class CandleAnatomyTransformer(BaseEstimator, TransformerMixin):
    """
    Candle Anatomy (sekcja 3):
    - body_ratio
    - upper_wick_ratio
    - lower_wick_ratio

    Wszystko normalizowane przez full_range = high-low.
    Edge-case high==low:
      - policy 'nan' (rekomendowane): ratios -> NaN (a później usuwamy wiersze/okna)
      - policy 'zeros': ratios -> 0.0
    """
    def __init__(
        self,
        open_col="open",
        high_col="high",
        low_col="low",
        close_col="close",
        eps: float = 1e-12,
        zero_range_policy: str = "nan",  # "nan" or "zeros"
        drop_na: bool = False,
    ):
        self.open_col = open_col
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col
        self.eps = eps
        self.zero_range_policy = zero_range_policy
        self.drop_na = drop_na

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")

        for col in [self.open_col, self.high_col, self.low_col, self.close_col]:
            if col not in X.columns:
                raise ValueError(f"Missing column: {col}")

        o = X[self.open_col].astype(float)
        h = X[self.high_col].astype(float)
        l = X[self.low_col].astype(float)
        c = X[self.close_col].astype(float)

        full_range_raw = (h - l)
        zero_range = full_range_raw <= self.eps

        # dzielnik: NaN na zero-range, żeby nie robić sztucznej geometrii
        denom = full_range_raw.mask(zero_range, np.nan)

        out = pd.DataFrame(index=X.index)

        out["body_ratio"] = (c - o).abs() / denom
        out["upper_wick_ratio"] = (h - np.maximum(o, c)) / denom
        out["lower_wick_ratio"] = (np.minimum(o, c) - l) / denom

        if self.zero_range_policy == "zeros":
            out.loc[zero_range, ["body_ratio", "upper_wick_ratio", "lower_wick_ratio"]] = 0.0
        elif self.zero_range_policy == "nan":
            # zostawiamy NaN
            pass
        else:
            raise ValueError("zero_range_policy must be 'nan' or 'zeros'")

        if self.drop_na:
            out = out.dropna()

        return out


In [72]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureConcatenator(BaseEstimator, TransformerMixin):
    """Łączy oryginalne X z nowymi cechami zwróconymi przez inny transformer."""
    def __init__(self, transformers: list[tuple[str, TransformerMixin]], drop_original: bool = True):
        self.transformers = transformers
        self.drop_original = drop_original

    def fit(self, X, y=None):
        for _, tr in self.transformers:
            tr.fit(X, y)
        return self

    def transform(self, X):
        feats = []
        for _, tr in self.transformers:
            feats.append(tr.transform(X))
        F = pd.concat(feats, axis=1)

        if self.drop_original:
            return F

        return pd.concat([X, F], axis=1)


In [73]:
price_tr = PriceTransformTransformer(drop_na=False)
anatomy_tr = CandleAnatomyTransformer(zero_range_policy="nan", drop_na=False)

feat_builder = FeatureConcatenator([
    ("price", price_tr),
    ("anatomy", anatomy_tr),
], drop_original=True)

X_feat = feat_builder.fit_transform(df)

print(X_feat.head())
print(X_feat.columns)


            log_return  hl_range  co_return  body_ratio  upper_wick_ratio  \
Date                                                                        
1952-01-02         NaN       0.0        0.0         NaN               NaN   
1952-01-03    0.003356       0.0        0.0         NaN               NaN   
1952-01-04    0.001674       0.0        0.0         NaN               NaN   
1952-01-07   -0.000418       0.0        0.0         NaN               NaN   
1952-01-08   -0.003771       0.0        0.0         NaN               NaN   

            lower_wick_ratio  
Date                          
1952-01-02               NaN  
1952-01-03               NaN  
1952-01-04               NaN  
1952-01-07               NaN  
1952-01-08               NaN  
Index(['log_return', 'hl_range', 'co_return', 'body_ratio', 'upper_wick_ratio',
       'lower_wick_ratio'],
      dtype='object')


In [74]:
# 1) brak absolutnych cen (kontrakt)
forbidden = {"open","high","low","close","adj_close"}
assert forbidden.isdisjoint(set(X_feat.columns))

# 2) log_return ma NaN w pierwszym wierszu (diff)
assert pd.isna(X_feat["log_return"].iloc[0])

# 3) geometria świecy sumuje się do 1 dla wierszy z policzalnym range
mask = X_feat[["body_ratio","upper_wick_ratio","lower_wick_ratio"]].notna().all(axis=1)
s = X_feat.loc[mask, "body_ratio"] + X_feat.loc[mask, "upper_wick_ratio"] + X_feat.loc[mask, "lower_wick_ratio"]
assert np.allclose(s.to_numpy(), 1.0, atol=1e-6)

print(f"✅ Price Transforms + Candle Anatomy: PASS (checked {mask.sum()} rows, skipped {(~mask).sum()} zero-range rows)")


✅ Price Transforms + Candle Anatomy: PASS (checked 16090 rows, skipped 2553 zero-range rows)
